In [ ]:

# ============================================================
# AIRLINE SATISFACTION — EXPLORATORY DATA ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")

# ============================================================
# 1. LOAD DATA
# ============================================================

ROOT = Path.cwd()
if not (ROOT / "data/raw/train.csv").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Submission shape:", submission.shape)

print("\nFirst 5 rows:")
display(train.head())

print("\nColumn types:")
display(train.dtypes.to_frame("dtype"))

print("\nNumerical summary:")
display(train.describe().T)

# ============================================================
# 2. DATA QUALITY
# ============================================================

print("\n" + "=" * 60)
print("2. DATA QUALITY")
print("=" * 60)

print("Duplicate rows:", train.duplicated().sum())
print("Duplicate IDs:", train["id"].duplicated().sum())

missing = pd.DataFrame({
    "Train missing": train.isna().sum(),
    "Test missing": test.isna().sum()
})

missing["Train missing %"] = (
    missing["Train missing"] / len(train) * 100
).round(3)

display(missing[missing.sum(axis=1) > 0])

# ============================================================
# 3. TARGET VARIABLE
# ============================================================

print("\n" + "=" * 60)
print("3. TARGET ANALYSIS")
print("=" * 60)

print("\nTarget distribution:")
display(train["satisfaction"].value_counts())

print("\nTarget percentages:")
display(
    train["satisfaction"].value_counts(normalize=True) * 100
)

fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(
    data=train,
    x="satisfaction",
    hue="satisfaction",
    palette="Set2",
    legend=False,
    ax=ax
)

ax.set_title("Passenger Satisfaction Distribution")
ax.set_xlabel("Satisfied")
ax.set_ylabel("Number of Passengers")

plt.tight_layout()
plt.show()

# ============================================================
# 4. CATEGORICAL FEATURES
# ============================================================

print("\n" + "=" * 60)
print("4. CATEGORICAL FEATURES")
print("=" * 60)

cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):

    rates = (
        train.groupby(col)["satisfaction"]
        .mean()
        .sort_values()
        * 100
    )

    rates.plot(
        kind="barh",
        ax=axes[i],
        color="steelblue"
    )

    axes[i].set_title(f"Satisfaction Rate by {col}")
    axes[i].set_xlabel("Satisfied (%)")
    axes[i].set_xlim(0, 100)

plt.tight_layout()
plt.show()

# ============================================================
# 5. NUMERICAL FEATURES
# ============================================================

print("\n" + "=" * 60)
print("5. NUMERICAL DISTRIBUTIONS")
print("=" * 60)

numeric_cols = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):

    # Clip extreme values for clearer visualisation only
    upper = train[col].quantile(0.99)
    values = train[col].clip(upper=upper)

    sns.histplot(
        values,
        bins=40,
        ax=axes[i],
        color="steelblue"
    )

    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

# ============================================================
# 6. SERVICE RATINGS VS SATISFACTION
# ============================================================

print("\n" + "=" * 60)
print("6. SERVICE RATINGS")
print("=" * 60)

rating_cols = [
    "Online boarding",
    "Inflight wifi service",
    "Seat comfort",
    "Inflight entertainment"
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, col in enumerate(rating_cols):

    rates = (
        train.groupby(col)["satisfaction"]
        .mean()
        .sort_index()
        * 100
    )

    axes[i].plot(
        rates.index,
        rates.values,
        marker="o",
        linewidth=2
    )

    axes[i].set_title(f"Satisfaction vs {col}")
    axes[i].set_xlabel("Service Rating")
    axes[i].set_ylabel("Satisfied (%)")
    axes[i].set_ylim(0, 100)

plt.tight_layout()
plt.show()

# ============================================================
# 7. FEATURE CORRELATIONS
# ============================================================

print("\n" + "=" * 60)
print("7. CORRELATIONS WITH TARGET")
print("=" * 60)

numeric_data = train.select_dtypes(include="number").copy()

# Convert Boolean target to 0/1
numeric_data["satisfaction"] = train["satisfaction"].astype(int)

correlations = (
    numeric_data
    .drop(columns="id", errors="ignore")
    .corr()["satisfaction"]
    .drop("satisfaction")
    .sort_values()
)

display(correlations.to_frame("Correlation"))

plt.figure(figsize=(10, 8))

correlations.plot(
    kind="barh",
    color=[
        "tomato" if x < 0 else "seagreen"
        for x in correlations
    ]
)

plt.title("Numerical Feature Correlation with Satisfaction")
plt.xlabel("Pearson Correlation")
plt.axvline(0, color="black", linewidth=1)

plt.tight_layout()
plt.show()

# ============================================================
# 8. TRAIN VS TEST CONSISTENCY
# ============================================================

print("\n" + "=" * 60)
print("8. TRAIN / TEST CONSISTENCY")
print("=" * 60)

train_features = set(train.columns) - {"satisfaction", "id"}
test_features = set(test.columns) - {"id"}

print("Features only in train:", train_features - test_features)
print("Features only in test:", test_features - train_features)

print("\nCategorical values:")
for col in cat_cols:
    print(f"\n{col}")
    print("Train:", sorted(train[col].dropna().unique()))
    print("Test: ", sorted(test[col].dropna().unique()))

# ============================================================
# 9. EDA SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("EDA COMPLETE")
print("=" * 60)

print(f"Training rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Missing values: {train.isna().sum().sum():,}")
print(f"Satisfaction rate: {train['satisfaction'].mean():.2%}")

print("\nTop positive numerical correlations:")
print(correlations.sort_values(ascending=False).head(5))

print("\nTop negative numerical correlations:")
print(correlations.head(5))
